# Training Linear Model

- Team: Voting Demographics
- Model Used: Logistic Regression

In [12]:
## Import packages and data
import math
import pandas as pd
import numpy as np
import itertools
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from data_cleaning import cleaned_data

In [13]:
df = cleaned_data()
df.head()

,VOTED,VOTEHOW,AGE,SEX,RACE,EDUC,NATIVITY,REGION,FAMINC
0,voted,in_person,elderly,male,white,hs_grad,native_born,south,lower
1,voted,in_person,middle_age,female,white,hs_grad,native_born,south,lower
6,voted,by_mail,elderly,female,white,master_higher,native_born,west,middle
7,not_voted,not_voted,young,male,white,hs_grad,native_born,south,lower
8,not_voted,not_voted,middle_age,female,white,hs_grad,native_born,south,lower


In [14]:
## Referred to the code from HW2.
class Dataset:
    """
    Prepare the dataset into training, validation, and test sets.
    """
    def __init__(self, y_col, y_col_map, other_drop_col, random_state = 123):
        self.data = cleaned_data()
        ## y_col can be either "VOTED" or "VOTEHOW". The other column will be used as other_drop_col.
        y_df = self.data[y_col].map(y_col_map).to_numpy()
        X_df = pd.get_dummies(self.data.drop([y_col, other_drop_col], axis=1), drop_first=True)
        self.feature_order = X_df.columns
        X_df = X_df.to_numpy()

        ## training set 70%, validation set 10%, test set 20%
        self.train_x, self.rest_x, self.train_y, self.rest_y = train_test_split(
            X_df, y_df, test_size=0.3, random_state=random_state)
        self.val_x, self.test_x, self.val_y, self.test_y = train_test_split(
            self.rest_x, self.rest_y, test_size=(2/3), random_state=random_state)
        
        ## Appending biases
        self.train_x = np.concatenate((np.ones((self.train_x.shape[0], 1)), self.train_x), axis=1)
        self.val_x = np.concatenate((np.ones((self.val_x.shape[0], 1)), self.val_x), axis=1)
        self.test_x = np.concatenate((np.ones((self.test_x.shape[0], 1)), self.test_x), axis=1)

    @staticmethod
    def shuffle(X, y):
        """ Shuffle training data """
        shuffled_indices = np.random.permutation(len(y))
        return X[shuffled_indices], y[shuffled_indices]

In [15]:
## Check if the arrays are created correctly.
voted_map = {"voted": 1, "not_voted": 0}
dataset_handler = Dataset("VOTED", voted_map, "VOTEHOW")

print(dataset_handler.train_x.shape)
print(dataset_handler.train_x[:5])
print(dataset_handler.feature_order)

(43986, 16)
[[1. 1. 0. 0. 0. 0. 0. 0. 0. 1. 1. 0. 0. 0. 0. 0.]
 [1. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 1.]
 [1. 0. 1. 1. 0. 0. 0. 0. 0. 1. 0. 0. 0. 1. 1. 0.]
 [1. 1. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 1. 0.]
 [1. 1. 0. 1. 0. 0. 0. 0. 1. 0. 0. 1. 0. 0. 0. 1.]]
Index(['AGE_middle_age', 'AGE_elderly', 'SEX_female', 'RACE_black',
       'RACE_indian_aleut_eskimo', 'RACE_asian', 'RACE_others',
       'EDUC_college_grad', 'EDUC_master_higher', 'NATIVITY_foreign_born',
       'REGION_midwest', 'REGION_south', 'REGION_west', 'FAMINC_middle',
       'FAMINC_upper'],
      dtype='str')


In [16]:
## Referred to the code from HW2.
def sigmoid(score, threshold=20.0):
    """
    Sigmoid function with a threshold
    :param score: (float) A real valued number to convert into a number between 0 and 1
    :param threshold: (float) Prevent overflow of exp by capping activation at 20.
                    (e.g., scores higher than 20 are converted to 20, scores lower than -20 are converted to -20).

    :return: (float) sigmoid function result.
    """
    if abs(score) > threshold:
        if score > 0:
            score = threshold
        else:
            score = -threshold
    return 1/(1 + np.exp(-score))

In [17]:
## Referred to the code from HW2.
class LogReg:
    def __init__(self, num_features, eta):
        """
        Create a logistic regression classifier
        :param num_features: (int) The number of features (including bias)
        :param eta: (float) learning rate
        """
        self.w = np.zeros(num_features)
        self.eta = eta

    def progress(self, examples_x, examples_y):
        """
        Given a set of examples, compute the probability and accuracy
        :param examples_x: (2D np.ndarray) The features from the dataset to score
        :param examples_y: (1D np.ndarray) The labels from the dataset to score

        :return: (float, float) A tuple of (log probability, accuracy, precision)
        """
        logprob = 0.0
        num_right = 0
        predicted_positive = 0
        true_positive = 0
        
        for x_i, y in zip(examples_x, examples_y):
            p = sigmoid(self.w.dot(x_i))
            if y == 1:
                logprob += math.log(p)
            else:
                logprob += math.log(1.0 - p)

            # Is the prediction right or wong
            if abs(y - p) <= 0.5:
                num_right += 1
            
            # To calculate precision
            if p >= 0.5:
                predicted_positive += 1
                if y == 1:
                    true_positive += 1

        accuracy = num_right / len(examples_x)

        if predicted_positive == 0:
            precision = 0.0
        else:
            precision = true_positive / predicted_positive

        return logprob, accuracy, precision

    def sgd_update(self, x_i, y, lam=0.0):
        """
        Compute a stochastic gradient update to improve the log likelihood.
        :param x_i: (1D np.ndarray) The features of the example to take the gradient with respect to
        :param y: (float) The target output of the example to take the gradient with respect to
        :param lam: (float) regularization term. Default is zero; only used in Part 2D.

        :return: (1D np.ndarray) Return the new value of the regression coefficients (w)
        """
        z = np.dot(x_i, self.w)
        ## L2 regularization
        nll_driv = (sigmoid(z) - y) * x_i + 2 * lam * self.w
        ## No regularization for a bias term
        nll_driv[0] -= 2 * lam * self.w[0]
        self.w = self.w - self.eta * nll_driv
        return self.w

In [18]:
def train(epochs, eta, store_epoch, lam=0, decay=0):
    """
    Train a LogReg object for a set number of epochs with a given eta.

    :param epochs: (int) total number of training epochs
    :param eta: (float) learning rate
    :param store_epoch: (int) store training and test accuracies every store_epoch epochs
    :param lam: (float) weight given to regularization term. Default 0. Only used in Part 2D.
    :param decay: (float) Used to update learning rate during training (Part 3).
                  Equals 0 when learning rate is constant throughout training (Part 2).

    :return (train_accuracy_array, test_accuracy_array, learning_rates): tuple of (List, List, List)
        :train_accuracy_array: training accuracy after every store_epoch epochs
        :test_accuracy_array: test accuracy after every store_epoch epochs
        :learning_rates: learning rate after every store_epoch epochs. All values in this list
                         will be the same if decay = 0 (Only required for Part 2F)

        Example: With epochs = 30 and store_epoch = 10, only store accuracies after epochs = 10, 20, and 30.
    """

    voted_map = {"voted": 1, "not_voted": 0}
    dataset_handler = Dataset("VOTED", voted_map, "VOTEHOW")

    lr = LogReg(dataset_handler.train_x.shape[1], eta)

    train_accuracy_array = []
    test_accuracy_array = []

    train_precision_array = []
    test_precision_array = []

    learning_rates = []

    for epoch in range(epochs):
        X, y = Dataset.shuffle(dataset_handler.train_x, dataset_handler.train_y)

        for i in range(len(X)):
            lr.sgd_update(X[i], y[i], lam)

        lr.eta = lr.eta / (1 + decay * epoch)

        if (epoch + 1) % store_epoch == 0:
            train_result = lr.progress(X, y)
            test_result = lr.progress(dataset_handler.test_x, dataset_handler.test_y)

            train_accuracy_array.append(train_result[1])
            test_accuracy_array.append(test_result[1])

            train_precision_array.append(train_result[2])
            test_precision_array.append(test_result[2])

            learning_rates.append(lr.eta)

    return train_accuracy_array, test_accuracy_array, train_precision_array, test_precision_array, learning_rates, lr.w

In [19]:
eta = 1e-3
epochs = 300
store_epoch = 60
train_acc, test_acc, train_prec, test_prec, learning_rates, weight = train(epochs, eta, store_epoch, lam=1e-5)

for i in range(len(train_acc)):
    print("\ntrain accuracy after {} epochs: {}".format((i+1)*store_epoch, train_acc[i]))
    print("test accuracy after {} epochs: {}".format((i+1)*store_epoch, test_acc[i]))
    print("train precision after {} epochs: {}".format((i+1)*store_epoch, train_prec[i]))
    print("test precision after {} epochs: {}".format((i+1)*store_epoch, test_prec[i]))
    print("learning rates after {} epochs: {}".format((i+1)*store_epoch, learning_rates[i]))
print("\nweights: {}".format(weight))
print(dataset_handler.feature_order)


train accuracy after 60 epochs: 0.7711544582367117
test accuracy after 60 epochs: 0.7702100572883513
train precision after 60 epochs: 0.7910502958579881
test precision after 60 epochs: 0.7916738493363213
learning rates after 60 epochs: 0.001

train accuracy after 120 epochs: 0.7702678124857909
test accuracy after 120 epochs: 0.7717218332272437
train precision after 120 epochs: 0.7852734006245008
test precision after 120 epochs: 0.7871710247947872
learning rates after 120 epochs: 0.001

train accuracy after 180 epochs: 0.7710635202109762
test accuracy after 180 epochs: 0.7705283259070655
train precision after 180 epochs: 0.7902995720399429
test precision after 180 epochs: 0.790993468545892
learning rates after 180 epochs: 0.001

train accuracy after 240 epochs: 0.770608830082299
test accuracy after 240 epochs: 0.7716422660725653
train precision after 240 epochs: 0.7845251881876085
test precision after 240 epochs: 0.786232495360216
learning rates after 240 epochs: 0.001

train accuracy 

In [20]:
def cross_validate(k, epochs, eta, lam=0, decay=0):
    """
    Do cross validation for k iterations.

    :param epochs: (int) total number of training epochs
    :param eta: (float) learning rate
    :param lam: (float) weight given to regularization term. Default 0.
    :param decay: (float) Used to update learning rate during training.
                    Equals 0 when learning rate is constant throughout training.

    :return average_error: (float) average error from all folds
    """
    
    voted_map = {"voted": 1, "not_voted": 0}
    dataset_handler = Dataset("VOTED", voted_map, "VOTEHOW")

    lr = LogReg(dataset_handler.val_x.shape[1], eta)

    # access the validation set
    X, y = Dataset.shuffle(dataset_handler.val_x, dataset_handler.val_y)
    # split the validation set into k folds
    kf = KFold(n_splits=k)

    error_rates = []

    # iterate through the folds, training on k-1 and testing on 1
    for i, (train_index, test_index) in enumerate(kf.split(X)):
        # train the model on all folds except i
        for epoch in range(epochs):
            X_i = X[train_index]
            y_i = y[train_index]

            for i in range(len(X_i)):
                lr.sgd_update(X_i[i], y[i], lam)

            lr.eta = lr.eta / (1 + decay * epoch)

            # test the model on the ith fold and get the error rate
            test_result = lr.progress(X[test_index], y[test_index])
            error_rates.append(1 - test_result[1])

    # average the error rates from all folds
    average_error = sum(error_rates) / len(error_rates)

    return average_error

In [21]:
def find_hyperparameters(k, eta_list, lam_list, epochs_list=[50]):
    """
    Find the hyperparameters with the lowest error.

    :param k: (int) total number of folds and iterations
    :param eta_list: (list[float]) list of learning rates
    :param lam_list: (list[float]) list of lambdas
    :param epochs_list: (list[int]) lsit of numbers of epochs

    :return lowest_error: (tuple((epochs, eta, lam), error) lowest average error
    """

    hyperparam_combos = list(itertools.product(eta_list, lam_list, epochs_list))

    # create a dictionary to track the error rates for different combinations
    error_rates = {}

    for eta, lam, epochs in hyperparam_combos:
        # validate the model with the different combinations of eta and lambda
        error = cross_validate(k, epochs, eta, lam=lam)

        error_rates[(eta, lam, epochs)] = error

    lowest_error = min(error_rates.items(), key=lambda item: item[1])

    return lowest_error

In [22]:
k = 5
# epochs_list = [50, 100, 150, 200]
eta_list = [1e-3, 1e-4, 1e-5, 1e-6]
lam_list = [0, 0.1, 0.05, 0.01]

lowest_error = find_hyperparameters(k, eta_list, lam_list)
print(f"The eta, lam, and number of epochs with the lowest error are {lowest_error[0]} with an error rate of {lowest_error[1]}")

The eta, lam, and number of epochs with the lowest error are (0.001, 0, 50) with an error rate of 0.23340158298243213
